# Step 4 – Run the trained U-Nets on Nepal (Sept–Oct 2024)
Applies model **A (SAR only)** and **B (SAR + slope + HAND)** from notebook 02 to:
- **koshi_terai**, flood 1 Oct 2024 (real flood, flat terrain)
- **kathmandu**, flood 2 Oct 2024 (flood already gone, steep urban terrain = false-alarm test)

There are no ground-truth labels yet, so this step compares the models with each other and with the classic change-detection map (`quick_flood`), and measures **where** they predict water (slopes, layover/shadow).

Runtime: **GPU (T4)**. Uses the Google Drive account that holds both `Paper2_flood/02_terrain` and `Paper2_flood_nepal`.

In [ ]:
!pip -q install segmentation-models-pytorch rasterio
import os, time, math, subprocess, numpy as np, pandas as pd, rasterio, torch
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEV)

from google.colab import drive
drive.mount('/content/drive')
MODELS = '/content/drive/MyDrive/Paper2_flood/02_terrain'
NEPAL = '/content/drive/MyDrive/Paper2_flood_nepal'
OUT = '/content/drive/MyDrive/Paper2_flood/04_nepal_inference'; os.makedirs(OUT, exist_ok=True)
FILES = {'koshi_terai': 'koshi_terai_flood2024-10-01_pre2024-09-19_orbit12.tif',
         'kathmandu':   'kathmandu_flood2024-10-02_pre2024-09-20_orbit19.tif'}
for f in ['unet_A_sar.pt', 'unet_B_sar_terrain.pt']:
    assert os.path.exists(f'{MODELS}/{f}'), f'missing {MODELS}/{f} - is this the same Drive account as notebook 02?'
for f in FILES.values():
    assert os.path.exists(f'{NEPAL}/{f}'), f'missing {NEPAL}/{f} - files may be in the Earth Engine account Drive; share or move them'
print('all files found')

## 1. Rebuild the training normalisation
The models expect inputs scaled exactly as in notebook 02, so the training-set mean/std are recomputed from Sen1Floods11 (cached terrain chips are reused).

In [ ]:
ROOT = '/content/sen1floods11'; os.makedirs(ROOT, exist_ok=True)
B = 'gs://sen1floods11/v1.1'
if not os.path.exists(f'{ROOT}/S1Hand'):
    for p in ['data/flood_events/HandLabeled/S1Hand', 'splits/flood_handlabeled']:
        subprocess.run(['gsutil', '-m', '-q', 'cp', '-r', f'{B}/{p}', f'{ROOT}/'], check=True)
train = [f.replace('_S1Hand.tif', '') for f in
         pd.read_csv(f'{ROOT}/flood_handlabeled/flood_train_data.csv', header=None)[0]]
CACHE = f'{MODELS}/terrain_chips'

def prep(sar, slope, hand):
    sar = np.clip(np.nan_to_num(sar, nan=-50), -50, 1)
    t = np.stack([np.clip(np.nan_to_num(slope), 0, 60), np.log1p(np.clip(np.nan_to_num(hand), 0, 200))])
    return np.concatenate([sar, t]).astype('float32')

xs = []
for c in train:
    with rasterio.open(f'{ROOT}/S1Hand/{c}_S1Hand.tif') as s: sar = s.read().astype('float32')
    t = np.load(f'{CACHE}/{c}.npy'); xs.append(prep(sar, t[0], t[1]))
xs = np.stack(xs)
MEAN, STD = xs.mean((0, 2, 3)), xs.std((0, 2, 3)) + 1e-6
del xs
print('mean', MEAN.round(2), 'std', STD.round(2))

## 2. Load the two models

In [ ]:
def load_model(path, ch):
    m = smp.Unet('resnet34', encoder_weights=None, in_channels=ch, classes=2).to(DEV)
    m.load_state_dict(torch.load(path, map_location=DEV)); return m.eval()

MODELS_ = {'A_sar': (load_model(f'{MODELS}/unet_A_sar.pt', 2), [0, 1]),
           'B_sar_terrain': (load_model(f'{MODELS}/unet_B_sar_terrain.pt', 4), [0, 1, 2, 3])}

## 3. Sliding-window inference over the full scenes
512 × 512 tiles with 64 px overlap; overlapping predictions are averaged. Each model runs on the **flood** and the **pre-flood** image. Empty edge pixels are masked. Run time (flood image) is recorded.

In [ ]:
BANDS = ['flood_VV', 'flood_VH', 'pre_VV', 'pre_VH', 'slope', 'hand', 'perm_water', 'ls_mask', 'quick_flood']

@torch.no_grad()
def predict(model, x, tile=512, stride=448, batch=8):
    C, H, W = x.shape
    Hp = max(tile, math.ceil((H - tile) / stride) * stride + tile)
    Wp = max(tile, math.ceil((W - tile) / stride) * stride + tile)
    xp = np.pad(x, ((0, 0), (0, Hp - H), (0, Wp - W)), mode='edge')
    prob = np.zeros((Hp, Wp), 'float32'); cnt = np.zeros((Hp, Wp), 'float32')
    pos = [(r, c) for r in range(0, Hp - tile + 1, stride) for c in range(0, Wp - tile + 1, stride)]
    for i in range(0, len(pos), batch):
        pb = pos[i:i + batch]
        xb = torch.tensor(np.stack([xp[:, r:r+tile, c:c+tile] for r, c in pb])).to(DEV)
        p = torch.softmax(model(xb), 1)[:, 1].cpu().numpy()
        for (r, c), pi in zip(pb, p):
            prob[r:r+tile, c:c+tile] += pi; cnt[r:r+tile, c:c+tile] += 1
    return (prob / cnt)[:H, :W]

from scipy.ndimage import binary_erosion
scenes, timing = {}, []
for name, f in FILES.items():
    with rasterio.open(f'{NEPAL}/{f}') as s:
        a = s.read().astype('float32'); profile = s.profile
    b = dict(zip(BANDS, a))
    # empty export pixels (NaN or exactly 0 dB) at the scene edges are NOT water -> mask them
    valid = np.ones(a.shape[1:], bool)
    for k in ['flood_VV', 'flood_VH', 'pre_VV', 'pre_VH']:
        valid &= np.isfinite(b[k]) & (b[k] != 0)
    valid = binary_erosion(valid, iterations=3)
    scenes[name] = {'bands': b, 'profile': profile, 'valid': valid}
    for date in ['flood', 'pre']:
        x = prep(np.stack([b[f'{date}_VV'], b[f'{date}_VH']]), b['slope'], b['hand'])
        x = (x - MEAN[:, None, None]) / STD[:, None, None]
        x[:, ~valid] = 0                      # neutral value (= training mean) outside the scene
        for mname, (model, ch) in MODELS_.items():
            if DEV == 'cuda': torch.cuda.synchronize()
            t0 = time.time(); p = predict(model, x[ch])
            if DEV == 'cuda': torch.cuda.synchronize()
            dt = time.time() - t0
            p[~valid] = 0
            scenes[name][f'{mname}_{date}'] = p
            if date == 'flood':
                km2 = valid.sum() / 1e4
                timing.append({'area': name, 'model': mname, 'valid km²': round(km2), 'seconds': round(dt, 1),
                               's per 1000 km²': round(dt / km2 * 1000, 1)})
timing = pd.DataFrame(timing); timing

## 4. Save prediction GeoTIFFs (water probability × 100)

In [ ]:
for name, s in scenes.items():
    prof = s['profile'].copy(); prof.update(count=2, dtype='uint8', nodata=255, compress='deflate')
    with rasterio.open(f'{OUT}/{name}_water_prob_A_B.tif', 'w', **prof) as dst:
        dst.write(np.stack([np.round(s['A_sar_flood'] * 100), np.round(s['B_sar_terrain_flood'] * 100)]).astype('uint8'))
        dst.set_band_description(1, 'A_sar water prob %'); dst.set_band_description(2, 'B_sar_terrain water prob %')
print('saved to', OUT)

## 5. Where do the models put water?
Two views per method:
- **water on flood date** (not permanent water): what each method sees on 1/2 Oct.
- **new water** = water on the flood date but **not** on the pre-flood date. This is what change detection measures, so it is the fair comparison. It also removes persistent monsoon water, wet paddy and dark urban surfaces such as the airport runway.

Kathmandu: the flood was over, so new water there is mostly **false alarm** (lower = better). `agree with quick (IoU)` uses new water and is agreement, not accuracy. Empty scene-edge pixels are excluded.

In [ ]:
rows, NEW = [], {}
for name, s in scenes.items():
    b, v = s['bands'], s['valid']
    perm = b['perm_water'] > 0; steep = b['slope'] >= 5; ls = b['ls_mask'] > 0
    quick = (b['quick_flood'] > 0) & v
    methods = {'quick change detection': (quick, quick)}
    for m in ['A_sar', 'B_sar_terrain']:
        wf, wp = s[f'{m}_flood'] > 0.5, s[f'{m}_pre'] > 0.5
        methods[m] = (wf & ~perm, wf & ~wp & ~perm)
    for mname, (wf, new) in methods.items():
        NEW[(name, mname)] = new
        n = max(new.sum(), 1)
        rows.append({'area': name, 'method': mname,
                     'water on flood date km²': round(wf.sum() / 1e4, 2),
                     'new water km²': round(new.sum() / 1e4, 2),
                     '% new on slope ≥5°': round(100 * (new & steep).sum() / n, 1),
                     'new in layover/shadow km²': round((new & ls).sum() / 1e4, 2),
                     'agree with quick (IoU)': round((new & quick).sum() / max((new | quick).sum(), 1), 3)})
summary = pd.DataFrame(rows)
summary.to_csv(f'{OUT}/nepal_summary.csv', index=False); summary

## 6. Visual comparison

In [ ]:
def show(name, step=3):
    s = scenes[name]; b = s['bands']
    panels = [('flood VV (dB)', np.where(s['valid'], np.clip(b['flood_VV'], -25, 0), np.nan), 'gray'),
              ('quick change detection', NEW[(name, 'quick change detection')], 'Blues'),
              ('A: new water', NEW[(name, 'A_sar')], 'Blues'),
              ('B: new water', NEW[(name, 'B_sar_terrain')], 'Blues')]
    fig, ax = plt.subplots(1, 4, figsize=(20, 6))
    for a, (t, im, cm) in zip(ax, panels):
        a.imshow(im[::step, ::step], cmap=cm); a.set_title(f'{name}: {t}'); a.axis('off')
    plt.tight_layout(); plt.savefig(f'{OUT}/{name}_comparison_new_water.png', dpi=120)

for name in scenes: show(name)